# Stage 3: CPU Post-Processing for YOLO26 Solar Filament Segmentation

Load the checkpoint and split metadata exported by the GPU training notebook. Run tiled inference, post-processing selection, official PQ evaluation, and submission creation on CPU only.

Set DEBUG=True for detailed CPU inference and per-image diagnostics. Set it to False to keep INFO-level stage logs and reduce prediction output.


## 1. Pipeline stages

1. Run yolo26-crop-preprocessing.ipynb with Kaggle Accelerator set to None; it uses four CPU processes for crop generation.
2. Run yolo26-crop-training-pipeline.ipynb with Kaggle GPU T4 x2; it trains on both GPUs and does not preprocess crops.
3. Run this notebook with Accelerator set to None; tiled inference and evaluation are forced to CPU with a four-thread limit.

Attach the training output artifact folder. The notebook checks the exported checkpoint and filename-grouped split before evaluation.


## 2. Configuration
Attach the exported yolo26_crop_training_artifacts directory as a persistent Kaggle dataset, or set TRAINING_ARTIFACT_DIR to its path. The notebook loads yolo26s_crop_training.pt and split_manifest.csv from that directory. You may set CHECKPOINT_PATH or SPLIT_MANIFEST_PATH explicitly if they are stored elsewhere. Prediction caches remain configurable below.

Inference is forced to CPU and thread pools are capped at four threads. In Kaggle, select the CPU accelerator.

In [ ]:
from pathlib import Path
import os,logging

DEBUG = True
logging.basicConfig(level=logging.DEBUG if DEBUG else logging.INFO,format='%(asctime)s | %(levelname)s | %(name)s | %(message)s',force=True)
LOGGER=logging.getLogger('filament_postprocessing')
LOGGER.info('DEBUG=%s; post-processing log level=%s',DEBUG,logging.getLevelName(logging.getLogger().level))

CPU_THREADS = 4
for variable in ('OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    os.environ[variable] = str(CPU_THREADS)

TRAINING_ARTIFACT_DIR = None  # Set to the exported yolo26_crop_training_artifacts directory when needed.
CHECKPOINT_PATH = None        # Defaults to TRAINING_ARTIFACT_DIR/yolo26s_crop_training.pt.
SPLIT_MANIFEST_PATH = None    # Defaults to TRAINING_ARTIFACT_DIR/split_manifest.csv.
BASELINE_CHECKPOINT_PATH = None  # Optional checkpoint to compare on the same held-out images.
BASELINE_COMPARISON_PATH = None
TILE_SIZE = 640
TILE_STRIDE = 480
TILE_BATCH_SIZE = 4
IMAGE_SIZE = TILE_SIZE
NMS_IOU = 0.50
MAX_DETECTIONS = 100
RAW_CONFIDENCE_FLOOR = 0.05
INFERENCE_DEVICE = 'cpu'
SEED = 42
VALIDATION_FRACTION = 0.20
COMPONENT_MIN_AREA = 5

SAVE_PREDICTIONS = True
LOAD_CACHED_PREDICTIONS = True
KAGGLE_WORKING = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd()
CACHE_WRITE_ROOT = KAGGLE_WORKING / 'filament_prediction_cache'
CACHE_READ_ROOT = None
RESULTS_PATH = KAGGLE_WORKING / 'postprocessing_results.csv'
BEST_CONFIG_PATH = KAGGLE_WORKING / 'postprocessing_best_config.json'
BEST_PER_IMAGE_PATH = KAGGLE_WORKING / 'postprocessing_best_per_image.csv'
SUBMISSION_PATH = KAGGLE_WORKING / 'submission.csv'
EVALUATION_HEIGHT = 2048
EVALUATION_WIDTH = 2048
OFFICIAL_IOU_THRESHOLD = 0.50
LOGGER.info('Working: %s | inference device: %s | CPU threads: %s',KAGGLE_WORKING,INFERENCE_DEVICE,CPU_THREADS)
LOGGER.debug('Artifacts: %s | split manifest: %s',TRAINING_ARTIFACT_DIR,SPLIT_MANIFEST_PATH)
LOGGER.debug('Cache read: %s | write: %s',CACHE_READ_ROOT,CACHE_WRITE_ROOT)

## 3. Imports and dependency setup
Scikit-image is used for skeletonization in optional endpoint experiments. Install it only if the Kaggle runtime does not already provide it.

In [ ]:
import importlib.util
import subprocess
import sys

required_modules = {
    'ultralytics': 'ultralytics',
    'pycocotools': 'pycocotools',
    'sklearn': 'scikit-learn',
    'tqdm': 'tqdm',
    'threadpoolctl': 'threadpoolctl',
}
missing = [package for module, package in required_modules.items() if importlib.util.find_spec(module) is None]
if missing:
    print('Installing missing inference/evaluation dependencies:', missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', *missing])
else:
    print('YOLO and evaluation dependencies are available.')

In [ ]:
import csv
import gc
import hashlib
import json
import os
import sys
import time
import warnings
from collections import defaultdict, Counter
from copy import deepcopy
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from scipy import ndimage as ndi
from sklearn.model_selection import GroupShuffleSplit
from tqdm.auto import tqdm
from threadpoolctl import threadpool_limits
from IPython.display import display
from pycocotools import mask as mask_utils
mask_util = mask_utils  # Name used by the copied official evaluator.
from pycocotools.coco import COCO
from ultralytics import YOLO

try:
    from skimage.morphology import skeletonize
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', 'scikit-image'])
    from skimage.morphology import skeletonize

warnings.filterwarnings('ignore')
cv2.setNumThreads(CPU_THREADS)
torch.set_num_threads(CPU_THREADS)
try:
    torch.set_num_interop_threads(1)
except RuntimeError:
    pass
THREADPOOL_LIMITER = threadpool_limits(limits=CPU_THREADS)
if INFERENCE_DEVICE != 'cpu':
    raise ValueError('This notebook is configured for CPU-only inference.')
LOGGER.info('CPU inference configured | device=%s | PyTorch/OpenCV/threadpool threads=%s',INFERENCE_DEVICE,CPU_THREADS)
LOGGER.debug('Postprocessing runtime | Python=%s | PyTorch=%s | CUDA available=%s',sys.version.split()[0],torch.__version__,torch.cuda.is_available())

## 4. Dataset paths and exported physical-image split
Load the filename-grouped manifest created by the training notebook. Verify that its COCO image IDs and annotation-view IDs match this dataset, and confirm that all views of each physical filename share a split. The split is not recomputed here.

In [ ]:
def find_training_artifact_dir(explicit_dir=None):
    if explicit_dir is not None:
        path = Path(explicit_dir).expanduser().resolve()
        if not path.is_dir():
            raise FileNotFoundError(f'Training artifact directory missing: {path}')
        return path
    roots = [Path('/kaggle/input'), KAGGLE_WORKING, Path.cwd()]
    candidates = set()
    for root in roots:
        if not root.exists():
            continue
        direct = root / 'yolo26_crop_training_artifacts'
        if (direct / 'split_manifest.csv').is_file() and (direct / 'yolo26s_crop_training.pt').is_file():
            candidates.add(direct.resolve())
        try:
            for manifest in root.rglob('split_manifest.csv'):
                parent = manifest.parent
                if (parent / 'yolo26s_crop_training.pt').is_file():
                    candidates.add(parent.resolve())
        except OSError:
            pass
    candidates = sorted(candidates)
    if not candidates:
        raise FileNotFoundError(
            'Could not find exported yolo26_crop_training_artifacts containing split_manifest.csv '
            'and yolo26s_crop_training.pt. Attach the artifact directory or set TRAINING_ARTIFACT_DIR.'
        )
    if len(candidates) > 1:
        raise RuntimeError('Multiple crop-training artifact directories found; set TRAINING_ARTIFACT_DIR:\n' + '\n'.join(map(str, candidates)))
    return candidates[0]

DATA_ROOT = find_dataset_root()
LOGGER.info('Dataset root: %s',DATA_ROOT)
TRAIN_DIR = DATA_ROOT / 'train'
TRAIN_IMAGE_DIR = TRAIN_DIR / 'train_images'
TEST_IMAGE_DIR = DATA_ROOT / 'test' / 'test_images'
ANNOTATION_PATH = TRAIN_DIR / 'MAGFiLO_1.0_Annotations_kaggle2026_train.json'
coco = COCO(str(ANNOTATION_PATH))
records = pd.DataFrame([{'image_id': str(i), 'file_name': im['file_name']} for i, im in coco.imgs.items()])
TRAINING_ARTIFACT_DIR = find_training_artifact_dir(TRAINING_ARTIFACT_DIR)
LOGGER.info('Training artifact bundle: %s',TRAINING_ARTIFACT_DIR)
SPLIT_MANIFEST_PATH = Path(SPLIT_MANIFEST_PATH) if SPLIT_MANIFEST_PATH is not None else TRAINING_ARTIFACT_DIR / 'split_manifest.csv'
if not SPLIT_MANIFEST_PATH.is_file():
    raise FileNotFoundError(f'Exported split manifest missing: {SPLIT_MANIFEST_PATH}')
exported_split = pd.read_csv(SPLIT_MANIFEST_PATH, dtype={'file_name': str, 'image_id': str, 'annotation_view_id': str, 'split': str})
required_columns = {'file_name', 'image_id', 'annotation_view_id', 'split'}
if not required_columns.issubset(exported_split.columns):
    raise ValueError(f'Split manifest needs columns {sorted(required_columns)}; got {exported_split.columns.tolist()}')
if exported_split.image_id.duplicated().any():
    raise ValueError('Exported split manifest contains duplicate image IDs.')
if set(exported_split.split) != {'train', 'val'}:
    raise ValueError(f'Unexpected split values: {sorted(set(exported_split.split))}')
expected = records[['image_id', 'file_name']].copy()
manifest_pairs = exported_split[['image_id', 'file_name']].copy()
if set(map(tuple, expected.to_numpy())) != set(map(tuple, manifest_pairs.to_numpy())):
    raise ValueError('Exported split manifest image_id/file_name mapping does not match the current COCO JSON.')
expected_views = expected.copy()
expected_views['annotation_view_id'] = expected_views.image_id.map(lambda value: value.split('-', 1)[0])
view_check = exported_split.merge(expected_views, on=['image_id', 'file_name'], suffixes=('_saved', '_current'))
if not (view_check.annotation_view_id_saved == view_check.annotation_view_id_current).all():
    raise ValueError('Exported annotation-view identifiers do not match COCO image IDs.')
records = records.merge(exported_split[['image_id', 'split']], on='image_id', validate='one_to_one')
train_records = records.loc[records.split == 'train'].drop(columns='split').reset_index(drop=True)
val_records = records.loc[records.split == 'val'].drop(columns='split').reset_index(drop=True)
train_files, val_files = set(train_records.file_name), set(val_records.file_name)
if train_files & val_files:
    raise AssertionError('Physical image leakage across train/validation split.')
if exported_split.groupby('file_name').split.nunique().max() != 1:
    raise AssertionError('Annotation views for a physical filename have different split assignments.')
split_identifier = __import__('hashlib').sha256(
    exported_split[['file_name','split']].drop_duplicates().sort_values('file_name').to_csv(index=False).encode('utf-8')
).hexdigest()[:16]
LOGGER.info('Split manifest validated | identifier=%s | train images=%s | validation images=%s',split_identifier,len(train_files),len(val_files))
training_config_path = TRAINING_ARTIFACT_DIR / 'training_config.json'
if training_config_path.is_file():
    training_config = json.loads(training_config_path.read_text(encoding='utf-8'))
    saved_identifier = training_config.get('split_identifier')
    if saved_identifier and saved_identifier != split_identifier:
        raise ValueError(f'Split identifier mismatch: manifest={split_identifier}, training config={saved_identifier}')
val_cache_records = [
    {'file_name': str(name), 'image_ids': sorted(group.image_id.astype(str).tolist())}
    for name, group in val_records.groupby('file_name', sort=True)
]
test_paths = sorted([p for p in TEST_IMAGE_DIR.iterdir() if p.is_file() and p.suffix.lower() in {'.jpg','.jpeg'}], key=lambda p:p.name)
test_cache_records = [{'file_name': p.name, 'image_ids': []} for p in test_paths]
if not test_cache_records:
    raise FileNotFoundError(f'No .jpg/.jpeg images in {TEST_IMAGE_DIR}')
for item in val_cache_records + test_cache_records:
    if '_' in Path(item['file_name']).stem:
        raise ValueError(f'Underscore in stem conflicts with official ID parsing: {item["file_name"]}')
print(f'Data: {DATA_ROOT}')
print(f'Artifacts: {TRAINING_ARTIFACT_DIR} | checkpoint: yolo26s_crop_training.pt')
print(f'Split {split_identifier} | train: {len(train_records):,} views/{len(train_files):,} physical; validation: {len(val_records):,}/{len(val_cache_records):,}')
print(f'Test files: {len(test_cache_records):,}; train/validation overlap: {len(train_files & val_files)}')

## 5. Resolve the exported crop-training checkpoint
No training is performed here. The notebook loads the checkpoint exported from the training notebook and records its identity in prediction-cache manifests.

In [ ]:
def find_checkpoint(explicit_path=None):
    if explicit_path is not None:
        path = Path(explicit_path).expanduser()
        if not path.is_file():
            raise FileNotFoundError(f'Configured checkpoint missing: {path}')
        return path.resolve()
    path = TRAINING_ARTIFACT_DIR / 'yolo26s_crop_training.pt'
    if not path.is_file():
        raise FileNotFoundError(f'Exported training checkpoint missing: {path}')
    return path.resolve()

TRAINED_CHECKPOINT = find_checkpoint(CHECKPOINT_PATH)
stat = TRAINED_CHECKPOINT.stat()
CHECKPOINT_IDENTITY = {'path': str(TRAINED_CHECKPOINT), 'size_bytes': int(stat.st_size), 'mtime_ns': int(stat.st_mtime_ns)}
model = None

def ensure_model_loaded():
    global model
    if model is None:
        LOGGER.info('Loading YOLO checkpoint on CPU: %s',TRAINED_CHECKPOINT)
        model = YOLO(str(TRAINED_CHECKPOINT))
        if getattr(model, 'task', None) != 'segment':
            raise RuntimeError(f'Expected segmentation checkpoint; got task={getattr(model, "task", None)!r}')
    return model

LOGGER.info('Checkpoint: %s (%.1f MiB)',TRAINED_CHECKPOINT,CHECKPOINT_IDENTITY['size_bytes']/2**20)

## 6. CPU tiled inference and raw prediction cache
Each original image is covered by overlapping 640x640 tiles with a 480-pixel stride and edge coverage. Grayscale tiles are converted to three identical channels. Raw tile-local masks, confidence, class, full-image boxes, and tile origins are cached before stitching; caches resume by physical image. All YOLO calls explicitly use CPU.


In [ ]:
def to_yolo_image(image):
    if image.ndim != 2:
        raise ValueError(f'Expected grayscale image, got {image.shape}')
    return cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)

def tile_starts(length, tile_size=TILE_SIZE, stride=TILE_STRIDE):
    if length <= tile_size:
        return [int((length - tile_size) // 2)]
    last = length - tile_size
    starts = list(range(0, last + 1, stride))
    if starts[-1] != last:
        starts.append(last)
    return starts

def extract_tile(image, x0, y0):
    h, w = image.shape
    tile = np.zeros((TILE_SIZE, TILE_SIZE), dtype=image.dtype)
    sx0, sy0 = max(0, x0), max(0, y0)
    sx1, sy1 = min(w, x0 + TILE_SIZE), min(h, y0 + TILE_SIZE)
    if sx1 > sx0 and sy1 > sy0:
        dx, dy = sx0 - x0, sy0 - y0
        tile[dy:dy + sy1 - sy0, dx:dx + sx1 - sx0] = image[sy0:sy1, sx0:sx1]
    return tile

def infer_raw_predictions(image, file_name, image_ids):
    h, w = image.shape
    locations = [(x, y) for y in tile_starts(h) for x in tile_starts(w)]
    LOGGER.debug('CPU tiled inference | image=%s | shape=%s | tiles=%s | tile batch=%s',file_name,(h,w),len(locations),TILE_BATCH_SIZE)
    tiles = [to_yolo_image(extract_tile(image, x, y)) for x, y in locations]
    local_masks, confidences, class_ids, boxes, tile_coordinates = [], [], [], [], []
    yolo = ensure_model_loaded()
    for offset in range(0, len(tiles), TILE_BATCH_SIZE):
        batch_tiles = tiles[offset:offset + TILE_BATCH_SIZE]
        results = yolo.predict(
            source=batch_tiles, imgsz=TILE_SIZE, conf=RAW_CONFIDENCE_FLOOR,
            iou=NMS_IOU, retina_masks=True, device='cpu', batch=len(batch_tiles),
            max_det=MAX_DETECTIONS, verbose=DEBUG,
        )
        for result, (x0, y0) in zip(results, locations[offset:offset + len(batch_tiles)]):
            if result.masks is None or result.boxes is None:
                continue
            masks = result.masks.data.detach().float().cpu().numpy()
            if masks.ndim == 2:
                masks = masks[None, ...]
            if masks.shape[1:] != (TILE_SIZE, TILE_SIZE):
                masks = np.stack([cv2.resize(m, (TILE_SIZE, TILE_SIZE), interpolation=cv2.INTER_NEAREST) for m in masks])
            masks = masks >= 0.5
            conf = result.boxes.conf.detach().float().cpu().numpy().astype(np.float32)
            classes = result.boxes.cls.detach().cpu().numpy().astype(np.int32)
            local_boxes = result.boxes.xyxy.detach().float().cpu().numpy().astype(np.float32)
            if not (len(masks) == len(conf) == len(classes) == len(local_boxes)):
                raise ValueError(f'Inconsistent YOLO tile arrays for {file_name}')
            local_masks.extend(masks)
            confidences.extend(conf)
            class_ids.extend(classes)
            global_boxes = local_boxes.copy()
            global_boxes[:, [0, 2]] += x0
            global_boxes[:, [1, 3]] += y0
            global_boxes[:, [0, 2]] = np.clip(global_boxes[:, [0, 2]], 0, w)
            global_boxes[:, [1, 3]] = np.clip(global_boxes[:, [1, 3]], 0, h)
            boxes.extend(global_boxes)
            tile_coordinates.extend([(x0, y0)] * len(masks))
    masks = np.asarray(local_masks, dtype=bool) if local_masks else np.zeros((0, TILE_SIZE, TILE_SIZE), dtype=bool)
    conf = np.asarray(confidences, dtype=np.float32)
    classes = np.asarray(class_ids, dtype=np.int32)
    boxes = np.asarray(boxes, dtype=np.float32).reshape(-1, 4)
    tile_coordinates = np.asarray(tile_coordinates, dtype=np.int32).reshape(-1, 2)
    areas = masks.reshape(len(masks), -1).sum(axis=1).astype(np.int32) if len(masks) else np.zeros(0, dtype=np.int32)
    return {
        'file_name': str(file_name), 'image_id': Path(file_name).stem,
        'image_ids': list(map(str, image_ids)), 'image_shape': (h, w),
        'masks': masks, 'confidences': conf, 'class_ids': classes,
        'boxes_xyxy': boxes, 'mask_areas': areas, 'tile_coordinates': tile_coordinates,
    }

def cache_file_path(cache_dir, file_name):
    key = hashlib.sha1(str(file_name).encode('utf-8')).hexdigest()
    return Path(cache_dir) / f'{key}.npz'

def write_prediction_cache(cache_dir, raw):
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    masks = raw['masks'].astype(bool, copy=False)
    n, tile_h, tile_w = masks.shape
    packed = np.packbits(masks.reshape(n, tile_h * tile_w), axis=1, bitorder='little') if n else np.zeros((0, (tile_h*tile_w+7)//8), dtype=np.uint8)
    h, w = raw['image_shape']
    target = cache_file_path(cache_dir, raw['file_name'])
    temp = target.with_name(target.stem + '.tmp.npz')
    np.savez_compressed(
        temp, packed_masks=packed, confidences=raw['confidences'],
        class_ids=raw['class_ids'], boxes_xyxy=raw['boxes_xyxy'],
        mask_areas=raw['mask_areas'], tile_coordinates=raw['tile_coordinates'],
        image_shape=np.asarray([h, w], dtype=np.int32), tile_shape=np.asarray([tile_h,tile_w],dtype=np.int32),
        file_name=np.asarray(raw['file_name']), image_id=np.asarray(raw['image_id']),
        image_ids_json=np.asarray(json.dumps(raw['image_ids'])),
    )
    os.replace(temp, target)
    return target

def read_prediction_cache(cache_dir, file_name):
    path = cache_file_path(cache_dir, file_name)
    if not path.is_file():
        raise FileNotFoundError(f'Missing cache entry: {path}')
    with np.load(path, allow_pickle=False) as data:
        h, w = map(int, data['image_shape'])
        tile_h, tile_w = map(int, data['tile_shape'])
        packed = data['packed_masks']
        n = len(packed)
        masks = np.unpackbits(packed, axis=1, count=tile_h*tile_w, bitorder='little').reshape(n,tile_h,tile_w).astype(bool) if n else np.zeros((0,tile_h,tile_w),dtype=bool)
        conf = data['confidences'].astype(np.float32, copy=True)
        classes = data['class_ids'].astype(np.int32, copy=True)
        boxes = data['boxes_xyxy'].astype(np.float32, copy=True)
        areas = data['mask_areas'].astype(np.int32, copy=True)
        tile_coordinates = data['tile_coordinates'].astype(np.int32,copy=True)
        saved_name = str(data['file_name'].item())
        image_id = str(data['image_id'].item())
        image_ids = json.loads(str(data['image_ids_json'].item()))
    if saved_name != str(file_name):
        raise ValueError(f'Cache key mismatch: requested {file_name}, found {saved_name}')
    instances = [
        {'mask': masks[i], 'confidence': float(conf[i]), 'class_id': int(classes[i]),
         'bbox_xyxy': boxes[i].copy(), 'area': int(areas[i]),
         'tile_origin': tuple(map(int,tile_coordinates[i])), 'tile_shape': (tile_h,tile_w)}
        for i in range(n)
    ]
    return {'file_name': saved_name, 'image_id': image_id, 'image_ids': image_ids,
            'image_shape': (h, w), 'instances': instances}

def cache_metadata(split, split_records):
    return {
        'schema_version': 2, 'split': split,
        'records': [{'file_name': r['file_name'], 'image_ids': sorted(map(str, r.get('image_ids', [])))} for r in split_records],
        'checkpoint': CHECKPOINT_IDENTITY,
        'inference': {'image_size': TILE_SIZE, 'tile_stride': TILE_STRIDE,
                      'tile_batch_size': TILE_BATCH_SIZE,'confidence_floor': RAW_CONFIDENCE_FLOOR,
                      'nms_iou': NMS_IOU, 'max_detections_per_tile': MAX_DETECTIONS,
                      'retina_masks': True, 'input_channels': 3, 'device': 'cpu'},
    }

def write_manifest(cache_dir, manifest):
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    target = cache_dir / 'cache_manifest.json'
    temp = cache_dir / 'cache_manifest.tmp.json'
    temp.write_text(json.dumps(manifest, indent=2, sort_keys=True), encoding='utf-8')
    os.replace(temp, target)

def read_manifest(cache_dir):
    path = Path(cache_dir) / 'cache_manifest.json'
    return json.loads(path.read_text(encoding='utf-8')) if path.is_file() else None

def cache_is_complete(cache_dir, expected):
    manifest = read_manifest(cache_dir)
    if manifest is None or manifest.get('metadata') != expected:
        return False
    names = {r['file_name'] for r in expected['records']}
    return names <= set(manifest.get('completed', [])) and all(cache_file_path(cache_dir, n).is_file() for n in names)

def ensure_prediction_cache(split, split_records):
    expected = cache_metadata(split, split_records)
    candidates = ([Path(CACHE_READ_ROOT) / split] if CACHE_READ_ROOT is not None else [])
    write_dir = Path(CACHE_WRITE_ROOT) / split
    candidates.append(write_dir)
    if LOAD_CACHED_PREDICTIONS:
        for candidate in candidates:
            if cache_is_complete(candidate, expected):
                LOGGER.info('Using complete %s cache: %s',split,candidate)
                return candidate
    if not SAVE_PREDICTIONS:
        raise FileNotFoundError(f'No matching {split} cache. Set SAVE_PREDICTIONS=True to create it.')
    write_dir.mkdir(parents=True, exist_ok=True)
    manifest = read_manifest(write_dir) if LOAD_CACHED_PREDICTIONS else None
    if manifest is not None and manifest.get('metadata') != expected:
        raise ValueError(f'{split} cache metadata mismatch. Change CACHE_WRITE_ROOT or set LOAD_CACHED_PREDICTIONS=False.')
    if manifest is None:
        manifest = {'metadata': expected, 'completed': []}
        write_manifest(write_dir, manifest)
    completed = set(manifest.get('completed', []))
    LOGGER.info('Creating %s raw tiled cache for %s physical images | CPU threads=%s',split,len(split_records),CPU_THREADS)
    for i, record in enumerate(tqdm(split_records,desc=f'Build {split} cache',disable=not DEBUG), start=1):
        name = record['file_name']
        LOGGER.debug('Cache item %s/%s | %s',i,len(split_records),name)
        if name in completed and cache_file_path(write_dir, name).is_file():
            continue
        image_dir = TRAIN_IMAGE_DIR if split == 'val' else TEST_IMAGE_DIR
        image = cv2.imread(str(image_dir / name), cv2.IMREAD_GRAYSCALE)
        if image is None:
            raise OSError(f'Could not read {image_dir / name}')
        raw = infer_raw_predictions(image, name, record.get('image_ids', []))
        write_prediction_cache(write_dir, raw)
        completed.add(name)
        manifest['completed'] = sorted(completed)
        write_manifest(write_dir, manifest)
        if i % 100 == 0 or i == len(split_records):
            LOGGER.info('%s cache progress: %s/%s images',split,i,len(split_records))
        del raw, image
        if i % 4 == 0:
            gc.collect()
    if not cache_is_complete(write_dir, expected):
        raise RuntimeError(f'{split} cache remains incomplete.')
    LOGGER.info('%s cache complete: %s',split,write_dir)
    return write_dir

## 7. Create or reuse validation predictions
A complete matching cache skips YOLO entirely. Otherwise validation inference runs once per physical image and writes a restartable cache.

In [ ]:
import time

# Start the timer
start_time = time.time()

VAL_CACHE_DIR = ensure_prediction_cache('val', val_cache_records)

# Calculate elapsed time
elapsed_time = time.time() - start_time
LOGGER.info('Validation cache ready: %s (%.2f sec)',VAL_CACHE_DIR,elapsed_time)


## 8. Standard representation and post-processing
Raw cache entries preserve each tile-local mask, confidence, class, full-image box, and tile origin. Post-processing first stitches likely duplicate detections from overlapping tiles using deterministic overlap and containment checks with a complete-link guard. It then applies the selected component or instance-preserving strategy. Raw cache arrays are never edited.

In [ ]:
def _bbox(mask):
    ys, xs = np.where(mask)
    return np.asarray([xs.min(), ys.min(), xs.max()+1, ys.max()+1], dtype=np.float32) if len(xs) else np.zeros(4, dtype=np.float32)

def _instance(mask, confidence, class_id, source_count=1):
    mask = np.asarray(mask, dtype=bool)
    return {'mask': mask, 'confidence': float(confidence), 'class_id': int(class_id),
            'bbox_xyxy': _bbox(mask), 'area': int(mask.sum()), 'source_count': int(source_count)}

def _apply_morphology(mask, cfg):
    out = np.asarray(mask, dtype=bool).copy()
    iterations = max(1, int(cfg.get('iterations', 1)))
    for key, op in [('opening_kernel', cv2.MORPH_OPEN), ('closing_kernel', cv2.MORPH_CLOSE)]:
        size = int(cfg.get(key, 0) or 0)
        if size:
            if size % 2 == 0:
                raise ValueError(f'{key} must be odd')
            kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (size, size))
            out = cv2.morphologyEx(out.astype(np.uint8), op, kernel, iterations=iterations) > 0
    if cfg.get('fill_holes', False):
        out = ndi.binary_fill_holes(out)
    return out

def _union_components(selected, shape, min_area):
    if not selected:
        return []
    union = np.zeros(shape, dtype=np.uint8)
    for item in selected:
        union |= item['mask'].astype(np.uint8)
    count, labels, stats, _ = cv2.connectedComponentsWithStats(union, connectivity=8)
    owners = defaultdict(list)
    for item in selected:
        for label in np.unique(labels[item['mask']]):
            if label:
                owners[int(label)].append(item)
    output = []
    for label in range(1, count):
        area = int(stats[label, cv2.CC_STAT_AREA])
        if area < min_area:
            continue
        mask = labels == label
        sources = owners.get(label, [])
        conf = max((s['confidence'] for s in sources), default=0.0)
        class_pixels = Counter()
        for source in sources:
            class_pixels[int(source['class_id'])] += int(np.logical_and(source['mask'], mask).sum())
        class_id = class_pixels.most_common(1)[0][0] if class_pixels else 0
        output.append(_instance(mask, conf, class_id, max(1, len(sources))))
    return sorted(output, key=lambda x: (-x['area'], x['bbox_xyxy'][1], x['bbox_xyxy'][0]))

def _endpoint_features(mask, tangent_radius=10):
    mask = np.asarray(mask, dtype=bool)
    if not mask.any():
        return []
    skel = skeletonize(mask)
    skel8 = skel.astype(np.uint8)
    neighbors = cv2.filter2D(skel8, cv2.CV_16S, np.ones((3, 3), dtype=np.uint8)) - skel8.astype(np.int16)
    endpoints = np.argwhere(skel & (neighbors == 1))
    if len(endpoints) == 0:
        return []
    coords = np.argwhere(skel)
    distance = cv2.distanceTransform(mask.astype(np.uint8), cv2.DIST_L2, 3)
    features = []
    radius2 = tangent_radius ** 2
    for y, x in endpoints:
        delta = coords - np.asarray([y, x])
        local = coords[(delta * delta).sum(axis=1) <= radius2]
        local = local[np.any(local != [y, x], axis=1)]
        if len(local) == 0:
            continue
        outward = np.asarray([y, x], dtype=np.float32) - local.mean(axis=0)
        norm = np.linalg.norm(outward)
        if norm < 1e-6:
            continue
        features.append({'xy': np.asarray([x, y], dtype=np.float32),
                         'direction_yx': outward / norm,
                         'width': max(1.0, float(2 * distance[int(y), int(x)]))})
    return features

def _compatible_pair(a, fa, b, fb, cfg):
    if not fa or not fb or a['class_id'] != b['class_id']:
        return None
    if min(a['confidence'], b['confidence']) < float(cfg.get('min_confidence', 0)):
        return None
    best = None
    for ea in fa:
        for eb in fb:
            delta = eb['xy'] - ea['xy']
            dist = float(np.linalg.norm(delta))
            if dist <= 0 or dist > float(cfg.get('max_endpoint_distance', 0)):
                continue
            toward_b = np.asarray([delta[1], delta[0]], dtype=np.float32) / dist
            cosine = min(float(np.dot(ea['direction_yx'], toward_b)),
                         float(np.dot(eb['direction_yx'], -toward_b)))
            if cosine < float(cfg.get('min_direction_cosine', 1)):
                continue
            ratio = max(ea['width'], eb['width']) / max(1.0, min(ea['width'], eb['width']))
            if ratio > float(cfg.get('max_width_ratio', np.inf)):
                continue
            candidate = {'distance': dist, 'cosine': cosine,
                         'p1': np.round(ea['xy']).astype(int), 'p2': np.round(eb['xy']).astype(int),
                         'w1': ea['width'], 'w2': eb['width']}
            if best is None or (dist, -cosine) < (best['distance'], -best['cosine']):
                best = candidate
    return best

def _merge_components(instances, cfg):
    if len(instances) < 2 or not cfg.get('enabled', False):
        return instances
    features = [_endpoint_features(x['mask'], int(cfg.get('tangent_radius', 10))) for x in instances]
    links, candidates = {}, []
    for i in range(len(instances)):
        for j in range(i + 1, len(instances)):
            link = _compatible_pair(instances[i], features[i], instances[j], features[j], cfg)
            if link is not None:
                links[(i, j)] = link
                candidates.append((link['distance'], -link['cosine'], i, j))
    groups = [{i} for i in range(len(instances))]
    for _, _, i, j in sorted(candidates):
        ga = next(g for g in groups if i in g)
        gb = next(g for g in groups if j in g)
        if ga is gb:
            continue
        cross = [(min(a, b), max(a, b)) for a in ga for b in gb]
        if all(pair in links for pair in cross):
            groups.remove(ga)
            groups.remove(gb)
            groups.append(ga | gb)
    merged = []
    for group in groups:
        members = [instances[i] for i in sorted(group)]
        if len(members) == 1:
            merged.append(members[0])
        else:
            union = np.zeros_like(members[0]['mask'], dtype=bool)
            for item in members:
                union |= item['mask']
            merged.append(_instance(union, min(m['confidence'] for m in members),
                                    members[0]['class_id'], sum(m.get('source_count', 1) for m in members)))
    return sorted(merged, key=lambda x: (-x['area'], x['bbox_xyxy'][1], x['bbox_xyxy'][0]))

def _bridge_gaps(instances, cfg):
    if len(instances) < 2 or not cfg.get('enabled', False):
        return instances
    features = [_endpoint_features(x['mask'], int(cfg.get('tangent_radius', 10))) for x in instances]
    masks = [x['mask'].copy() for x in instances]
    for i in range(len(instances)):
        for j in range(i + 1, len(instances)):
            link = _compatible_pair(instances[i], features[i], instances[j], features[j], cfg)
            if link is None:
                continue
            p1, p2 = tuple(map(int, link['p1'])), tuple(map(int, link['p2']))
            thickness = max(1, int(round((link['w1'] + link['w2']) / 4)))
            for idx in (i, j):
                canvas = masks[idx].astype(np.uint8)
                cv2.line(canvas, p1, p2, 1, thickness=thickness, lineType=cv2.LINE_8)
                masks[idx] = canvas > 0
    return [_instance(mask, item['confidence'], item['class_id'], item.get('source_count', 1))
            for mask, item in zip(masks, instances)]

def _local_mask_bbox(instance):
    ys,xs=np.where(instance['mask'])
    if not len(xs): return np.zeros(4,dtype=np.float32)
    x0,y0=instance['tile_origin']
    return np.asarray([x0+xs.min(),y0+ys.min(),x0+xs.max()+1,y0+ys.max()+1],dtype=np.float32)

def _place_local_mask(instance, shape):
    h,w=shape;local=np.asarray(instance['mask'],dtype=bool);x0,y0=instance['tile_origin']
    out=np.zeros((h,w),dtype=bool)
    sx0,sy0=max(0,-x0),max(0,-y0);sx1,sy1=min(local.shape[1],w-x0),min(local.shape[0],h-y0)
    if sx1>sx0 and sy1>sy0:
        out[y0+sy0:y0+sy1,x0+sx0:x0+sx1]=local[sy0:sy1,sx0:sx1]
    return out

def _boxes_intersect(a,b):
    return a[0]<b[2] and b[0]<a[2] and a[1]<b[3] and b[1]<a[3]

def _tile_windows_overlap(a,b):
    ax,ay=a['tile_origin'];bx,by=b['tile_origin']
    ah,aw=a.get('tile_shape',(TILE_SIZE,TILE_SIZE));bh,bw=b.get('tile_shape',(TILE_SIZE,TILE_SIZE))
    return ax<bx+bw and bx<ax+aw and ay<by+bh and by<ay+ah

def _overlap_in_common_tile(a,b):
    ax,ay=a['tile_origin'];bx,by=b['tile_origin']
    ah,aw=a['mask'].shape;bh,bw=b['mask'].shape
    x0,y0=max(ax,bx),max(ay,by);x1,y1=min(ax+aw,bx+bw),min(ay+ah,by+bh)
    if x1<=x0 or y1<=y0:return 0,0,0
    am=a['mask'][y0-ay:y1-ay,x0-ax:x1-ax]
    bm=b['mask'][y0-by:y1-by,x0-bx:x1-bx]
    intersection=int(np.logical_and(am,bm).sum())
    area_a=int(a['mask'].sum());area_b=int(b['mask'].sum())
    union=area_a+area_b-intersection
    iou=intersection/union if union else 0.0
    containment=intersection/min(area_a,area_b) if min(area_a,area_b)>0 else 0.0
    return intersection,iou,containment

def stitch_tile_predictions(instances,image_shape,config):
    """Merge only cross-tile duplicates with substantial mask overlap; ties are deterministic."""
    if not instances:return []
    cfg={'enabled':True,'mask_iou_threshold':0.45,'containment_threshold':0.80,
         'min_intersection_pixels':25,**(config or {})}
    ordered=sorted(instances,key=lambda z:(z['tile_origin'][1],z['tile_origin'][0],
        int(z['class_id']),tuple(np.round(_local_mask_bbox(z)).astype(int)), -float(z['confidence'])))
    if not cfg.get('enabled',True):
        groups=[{i} for i in range(len(ordered))]
    else:
        links={};candidates=[]
        bboxes=[_local_mask_bbox(item) for item in ordered]
        for i in range(len(ordered)):
            a=ordered[i]
            for j in range(i+1,len(ordered)):
                b=ordered[j]
                if a['class_id']!=b['class_id'] or a['tile_origin']==b['tile_origin']:
                    continue
                if not _tile_windows_overlap(a,b) or not _boxes_intersect(bboxes[i],bboxes[j]):
                    continue
                inter,iou,cover=_overlap_in_common_tile(a,b)
                if inter>=int(cfg['min_intersection_pixels']) and (
                    iou>=float(cfg['mask_iou_threshold']) or cover>=float(cfg['containment_threshold'])):
                    links[(i,j)]=(iou,cover,inter)
                    candidates.append((-cover,-iou,-inter,i,j))
        groups=[{i} for i in range(len(ordered))]
        for _,_,_,i,j in sorted(candidates):
            ga=next(g for g in groups if i in g);gb=next(g for g in groups if j in g)
            if ga is gb:continue
            cross=[(min(a,b),max(a,b)) for a in ga for b in gb]
            if all(pair in links for pair in cross):
                groups.remove(ga);groups.remove(gb);groups.append(ga|gb)
    output=[]
    for group in sorted(groups,key=lambda g:min(g)):
        members=[ordered[i] for i in sorted(group)]
        full=np.zeros(image_shape,dtype=bool)
        for item in members:full|=_place_local_mask(item,image_shape)
        if not full.any():continue
        output.append({'mask':full,'confidence':max(m['confidence'] for m in members),
            'class_id':members[0]['class_id'],'area':int(full.sum()),'source_count':len(members),
            'tile_origins':sorted(set(m['tile_origin'] for m in members))})
    return sorted(output,key=lambda x:(-x['area'],_bbox(x['mask'])[1],_bbox(x['mask'])[0]))

def postprocess_predictions(predictions, config):
    h, w = predictions['image_shape']
    threshold = float(config.get('confidence', 0))
    raw_min_area = int(config.get('min_instance_area', 0))
    morph = config.get('morphology', {})
    selected = []
    for raw in predictions['instances']:
        if raw['confidence'] < threshold:
            continue
        mask = raw['mask'].astype(bool, copy=True)
        if morph:
            mask = _apply_morphology(mask, morph)
        area = int(mask.sum())
        if area == 0 or area < raw_min_area:
            continue
        selected.append({'mask': mask, 'confidence': raw['confidence'],
                         'class_id': raw['class_id'], 'area': area, 'tile_origin': raw.get('tile_origin', (0,0)), 'tile_shape': raw.get('tile_shape', (h,w))})
    stitched = stitch_tile_predictions(selected, (h, w), config.get('stitch', {}))
    if config.get('combine_mode', 'connected_components') == 'instances':
        output = [_instance(x['mask'], x['confidence'], x['class_id'], x.get('source_count', 1))
                  for x in stitched if int(x['mask'].sum()) >= int(config.get('component_min_area', COMPONENT_MIN_AREA))]
    else:
        output = _union_components(stitched, (h, w), int(config.get('component_min_area', COMPONENT_MIN_AREA)))
    if config.get('gap_bridge', {}).get('enabled', False):
        output = _bridge_gaps(output, config['gap_bridge'])
    if config.get('merge', {}).get('enabled', False):
        output = _merge_components(output, config['merge'])
    return output

def encode_mask_rle(binary_mask):
    rle = mask_utils.encode(np.asfortranarray(np.asarray(binary_mask, dtype=np.uint8)))
    return rle['counts'].decode('utf-8')

## 9. Experiment configurations
Threshold, area, morphology, merge, and bridge values are starting sweep points, not universal optima. Merge uses skeleton endpoints, facing orientation, endpoint width ratio, confidence and same-class gates, plus a complete-link guard. Gap bridging adds a bridge to both masks while preserving instance IDs. No robust split rule is available in the supplied workflow; the split placeholder is a no-op and is not scored.

In [ ]:
BASELINE_CONFIG = {
    'confidence': 0.25, 'min_instance_area': 0,
    'component_min_area': COMPONENT_MIN_AREA, 'morphology': {},
    'merge': {'enabled': False}, 'gap_bridge': {'enabled': False},
    'stitch': {'enabled': True, 'mask_iou_threshold': 0.45, 'containment_threshold': 0.80,
               'min_intersection_pixels': 25}, 'combine_mode': 'connected_components',
}

def make_config(**updates):
    config = deepcopy(BASELINE_CONFIG)
    for key, value in updates.items():
        if key in {'merge', 'gap_bridge', 'morphology', 'stitch'}:
            config[key].update(value)
        else:
            config[key] = value
    return config

EXPERIMENTS = [{'name': 'baseline', 'config': make_config()},
               {'name': 'stitched_instances', 'config': make_config(combine_mode='instances')},
               {'name': 'stitch_strict', 'config': make_config(stitch={'mask_iou_threshold': 0.60, 'containment_threshold': 0.90})},
               {'name': 'stitch_loose', 'config': make_config(stitch={'mask_iou_threshold': 0.35, 'containment_threshold': 0.70})}]
for conf in (0.05, 0.10, 0.15, 0.20, 0.30, 0.40, 0.50, 0.60):
    EXPERIMENTS.append({'name': f'conf_{conf:.2f}', 'config': make_config(confidence=conf)})
for area in (20, 50):
    EXPERIMENTS.append({'name': f'min_area_{area}', 'config': make_config(min_instance_area=area)})
EXPERIMENTS += [
    {'name': 'morph_open_3', 'config': make_config(morphology={'opening_kernel': 3, 'iterations': 1})},
    {'name': 'morph_close_3', 'config': make_config(morphology={'closing_kernel': 3, 'iterations': 1})},
    {'name': 'morph_fill_holes', 'config': make_config(morphology={'fill_holes': True})},
]
for distance, cosine in ((10, 0.80), (15, 0.80), (25, 0.90)):
    EXPERIMENTS.append({
        'name': f'merge_{distance}px_cos_{cosine:.2f}',
        'config': make_config(merge={'enabled': True, 'max_endpoint_distance': distance,
                                     'min_direction_cosine': cosine, 'min_confidence': 0.25,
                                     'max_width_ratio': 2.5, 'tangent_radius': 10}),
    })
for distance, cosine in ((10, 0.80), (15, 0.90)):
    EXPERIMENTS.append({
        'name': f'bridge_{distance}px_cos_{cosine:.2f}',
        'config': make_config(gap_bridge={'enabled': True, 'max_endpoint_distance': distance,
                                          'min_direction_cosine': cosine, 'min_confidence': 0.25,
                                          'max_width_ratio': 2.5, 'tangent_radius': 10}),
    })
if any(e['config']['confidence'] < RAW_CONFIDENCE_FLOOR for e in EXPERIMENTS):
    raise ValueError('An experiment confidence is below the raw cache floor.')
SPLIT_STRATEGY_PLACEHOLDER = {'enabled': False, 'behavior': 'no-op'}
LOGGER.info('%s post-processing experiments configured.',len(EXPERIMENTS))
LOGGER.debug('Experiment names: %s', [e['name'] for e in EXPERIMENTS])

## 10. Official competition evaluator
The next helper and PQ cells are copied verbatim from self-evaluation-notebook.ipynb. GT is encoded from COCO annToMask, and predictions use the same compressed COCO RLE representation.

In [ ]:
from matplotlib.patches import Patch

In [ ]:
# Helper functions

def get_overlap_matrices(
        gt_layers: torch.Tensor,
        pred_layers: torch.Tensor,
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Computes pairwise IoU and Dice scores between ground-truth (GT) and predicted mask layers.
    This is called for all segmentations corresponding to one image.

    Args:
        gt_layers: Ground-truth layers, shape (n_gt, H, W), values in {0, 1}.
        pred_layers: Prediction layers, shape (n_pred, H, W), values in {0, 1}.

    Returns:
        A tuple ``(iou_matrix, dice_matrix)`` of tensors, each of shape
        (n_gt, n_pred) with values in [0, 1], where:

            iou_matrix[i, j]  = IoU(GT layer i, predicted layer j)
            dice_matrix[i, j] = Dice(GT layer i, predicted layer j)

        - **Rows (i, size n_gt)**: ground-truth filaments
        - **Columns (j, size n_pred)**: predicted filaments

        Example for n_gt=2 GT filaments and n_pred=3 predictions:

            iou_matrix = [[IoU(gt_0, pred_0), IoU(gt_0, pred_1), IoU(gt_0, pred_2)],
                          [IoU(gt_1, pred_0), IoU(gt_1, pred_1), IoU(gt_1, pred_2)]]
    """
    n_gt, height, width = gt_layers.shape
    n_pred = pred_layers.shape[0]

    gt_flat = gt_layers.reshape(n_gt, height * width)
    pred_flat = pred_layers.reshape(n_pred, height * width)

    intersection = torch.matmul(gt_flat, pred_flat.t())

    gt_areas = gt_flat.sum(dim=1).view(-1, 1)
    pred_areas = pred_flat.sum(dim=1).view(1, -1)

    union = gt_areas + pred_areas - intersection

    iou_matrix = torch.where(
        union == 0,
        torch.tensor(0., device=gt_layers.device),
        intersection / union,
    )

    dice_matrix = torch.where(
        union == 0,
        torch.tensor(0., device=gt_layers.device),
        2 * intersection / (gt_areas + pred_areas),
    )

    return iou_matrix, dice_matrix

def fp_count_hit(hit_matrix: torch.Tensor) -> int:
    """
    Counts false-positive predictions for one image from a matching matrix where
    the value is 1 if the predicted filament overlaps with the GT filament and 0 otherwise.

    A predicted filament is a false positive if it does not hit any GT filament:
    the sum of its column is zero.

    Example (rows = GT, columns = predictions):

        hit_matrix = [[0, 0, 1],
                      [0, 0, 0],
                      [0, 0, 1]]

        # column sums: [0, 0, 2]  ->  columns 0 and 1 are FPs
        fp_count_hit(hit_matrix) -> 2

    Args:
        hit_matrix: Boolean tensor of shape (n_gt, n_pred), match indicators per pair.

    Returns:
        Scalar integer tensor: number of predictions with no qualifying GT overlap.
    """
    gt_matches_per_pred = hit_matrix.sum(dim=0)
    return (gt_matches_per_pred == 0).sum().item()

def fn_count_hit(hit_matrix: torch.Tensor) -> int:
    """
    Counts false-negative annotations for one image from a matching matrix where
    the value is 1 if the predicted filament overlaps with the GT filament and 0 otherwise.

    A GT filament is a false negative if it does not hit any predicted filament: the sum of
    its row is zero.

    Example (rows = GT, columns = predictions):

        hit_matrix = [[0, 0, 1],
                      [0, 0, 0],
                      [0, 0, 1]]

        # row sums: [1, 0, 1]  ->  row 1 is FN
        fn_count_hit(hit_matrix) -> 1

    Args:
        hit_matrix: Boolean tensor of shape (n_gt, n_pred), match indicators per pair.

    Returns:
        Scalar integer tensor: number of annotations with no qualifying prediction overlap.
    """
    pred_matches_per_gt = hit_matrix.sum(dim=1)
    return (pred_matches_per_gt == 0).sum().item()

def rles_to_layers(rles: list[str], height: int = 2048, width: int = 2048) -> np.ndarray:
    """
    Decode a list of compressed COCO RLE strings into an (n_masks, H, W) binary mask stack.

    Args:
        rles: Compressed COCO RLE strings, one per filament.
        height: Mask height in pixels.
        width: Mask width in pixels.

    Returns:
        float32 array of shape (n_masks, H, W) with values in {0, 1}, where
        n_masks is ``len(rles)``. Returns an empty (0, H, W) array if ``rles``
        is empty.
    """
    if not rles:
        return np.zeros((0, height, width), dtype=np.float32)
    rle_dicts = [{"size": [height, width], "counts": rle} for rle in rles]
    # decode -> (H, W, n_masks), transpose to (n_masks, H, W)
    masks = mask_util.decode(rle_dicts)
    return masks.transpose(2, 0, 1).astype(np.float32)
    
def process_entry(
        annotator_image: str,
        gt_annotator_image_ids: pd.Series,
        pred_image_ids: pd.Series,
        gt_df: pd.DataFrame,
        pred_df: pd.DataFrame,
) -> tuple[torch.Tensor, torch.Tensor, int, int]:
    """
    Matches one annotator-image entry between prediction and GT and computes
    the overlap matrices between its GT and predicted filaments.

    Args:
        annotator_image: Annotator-image identifier in the form ``"<annotator_id>-<image_id>"``.
        gt_annotator_image_ids: Series of annotator-image identifiers, one per row of ``gt_df``.
        pred_image_ids: Series of image identifiers, one per row of ``pred_df``.
        gt_df: Ground-truth filaments; columns ``filament_id`` and ``segmentation_rle``.
        pred_df: Predicted filaments; same columns as ``gt_df``.

    Returns:
        Tuple ``(iou_matrix, dice_matrix, n_gt, n_pred)`` where the matrices
        have shape (n_gt, n_pred).
    """
    image_id = annotator_image.split("-", maxsplit=1)[1]

    # (n_gt, H, W) stack with one layer per GT filament of this annotator-image
    gt_rles = gt_df.loc[gt_annotator_image_ids == annotator_image, "segmentation_rle"].tolist()
    gt_layers = torch.from_numpy(rles_to_layers(gt_rles))

    # (n_pred, H, W) stack with one layer per predicted filament of this image
    pred_rles = pred_df.loc[pred_image_ids == image_id, "segmentation_rle"].tolist()
    pred_layers = torch.from_numpy(rles_to_layers(pred_rles))

    n_gt = len(gt_rles)
    n_pred = len(pred_rles)

    iou_matrix, dice_matrix = get_overlap_matrices(gt_layers, pred_layers)

    return iou_matrix, dice_matrix, n_gt, n_pred

def get_overlap_df(gt_df: pd.DataFrame, pred_df: pd.DataFrame) -> pd.DataFrame:
    """
    Build a per-annotator-image table of GT vs. prediction overlap matrices.

    Args:
        gt_df: Ground-truth filaments; columns ``filament_id`` and
            ``segmentation_rle``.
        pred_df: Predicted filaments; same columns as ``gt_df``.

    Returns:
        DataFrame with one row per annotator-image and columns:

        - ``annotator_image``: annotator-image identifier
        - ``iou_matrix``: IoU tensor of shape (n_gt, n_pred)
        - ``dice_matrix``: Dice tensor of shape (n_gt, n_pred)
        - ``n_gt``: number of GT filaments in the image
        - ``n_pred``: number of predicted filaments in the image
    """
    gt_annotator_image_ids = gt_df["filament_id"].str.split("_", n=1).str[0]
    pred_image_ids = pred_df["filament_id"].str.split("_", n=1).str[0]

    annotator_images = gt_annotator_image_ids.unique()

    overlap_df = pd.DataFrame({"annotator_image": annotator_images})

    results = overlap_df["annotator_image"].apply(
        process_entry,
        gt_annotator_image_ids=gt_annotator_image_ids,
        pred_image_ids=pred_image_ids,
        gt_df=gt_df,
        pred_df=pred_df,
    )

    overlap_df["iou_matrix"], overlap_df["dice_matrix"], overlap_df["n_gt"], overlap_df[
            "n_pred"] = zip(*results)

    return overlap_df

In [ ]:
def get_pq_score(overlap_df: pd.DataFrame) -> float:
    """
    Compute the Panoptic Quality (PQ) score over all images.

    A GT/prediction pair is a true positive (TP) when its IoU exceeds the
    IoU threshold (0.5). Predictions with no match are false positives (FP),
    GT filaments with no match are false negatives (FN), and:

        PQ = sum(IoU of TP pairs) / (|TP| + 0.5 * |FP| + 0.5 * |FN|)

    Args:
        overlap_df: Per-image overlap table as produced by ``get_overlap_df``;
            must contain columns ``iou_matrix`` (IoU tensor of shape
            (n_gt, n_pred)), ``n_gt`` (GT filament count), and ``n_pred``
            (predicted filament count).

    Returns:
        Panoptic Quality score in ``[0, 1]`` (higher is better), or ``0.0`` if
        the denominator is zero.
    """
    iou_threshold: float = 0.5

    tp_iou_scores: list[float] = []
    fp_count = 0
    fn_count = 0

    for row in overlap_df.itertuples(index=False):
        iou_matrix = row.iou_matrix
        n_gt = row.n_gt
        n_pred = row.n_pred

        if n_gt == 0:
            fp_count += n_pred
            continue

        if n_pred == 0:
            fn_count += n_gt
            continue

        hit_matrix = iou_matrix > iou_threshold

        tp_iou_scores.extend(iou_matrix[hit_matrix].tolist())

        fp_count += fp_count_hit(hit_matrix)
        fn_count += fn_count_hit(hit_matrix)

    tp_count = len(tp_iou_scores)
    denominator = tp_count + 0.5 * fp_count + 0.5 * fn_count

    if denominator > 0:
        pq_score = sum(tp_iou_scores) / denominator
    else:
        pq_score = 0.0

    return pq_score

The official notebook also plots overlap-score distributions and many-to-one/one-to-many relations. Those helpers are copied verbatim below. SQ/RQ are decompositions of the official TP IoU sum and official TP/FP/FN denominator; the PQ value itself is returned by the copied get_pq_score.

In [ ]:
def plot_distribution(matrices: pd.Series, title: str, x_label: str):
    """
    Plots histogram of pairwise overlap scores between GT and predicted filaments
    (scores > 0 only).

    Flattens all per-image score matrices into a single sample, drops
    non-overlapping (zero) pairs, and plots normalized bar frequencies with
    a mean line.

    Args:
        matrices: Iterable of per-image score matrices, e.g. the ``iou_matrix``
            or ``dice_matrix`` column of the dataframe returned by
            ``get_overlap_df``.
        title: Plot title.
        x_label: X-axis label.
    """

    # every entry is an (n_gt, n_pred) tensor with its own shape, so they cannot be stacked
    scores = np.concatenate(
        [np.asarray(m, dtype=float).ravel() for m in matrices] or [np.zeros(0)]
    )
    # non-overlapping pairs carry no information and would swamp the zero bin
    scores = scores[scores > 0]

    x_max = 1.0 if scores.size == 0 or scores.max() <= 1.0 else scores.max()

    if x_max <= 1.0:
        n_bins = 50
        bins = np.linspace(0.0, x_max, n_bins + 1)
        bar_width = 0.02
    else:
        bins = np.arange(0, int(x_max) + 2)
        bar_width = 0.8

    counts, edges = np.histogram(scores, bins=bins)
    probabilities = counts / counts.sum() if counts.sum() > 0 else counts
    centers = (edges[:-1] + edges[1:]) / 2.0

    plt.bar(centers, probabilities, width=bar_width, edgecolor="black", alpha=0.7, color="navy")
    plt.grid(axis="y", linestyle="--")

    if scores.size > 0:
        mean_score = scores.mean()
        plt.axvline(
            mean_score,
            color="crimson",
            linestyle="--",
            linewidth=1.5,
            label=f"Mean = {mean_score:.3f}",
        )
        plt.legend(fontsize=8)
    plt.xlabel(x_label)
    plt.ylabel("Probability")
    plt.title(title)


def plot_m2n_counts(overlap_df: pd.DataFrame, title: str = "Many to One and One to Many Distribution"):
    """
    Plots frequency of GT filaments and predictions matching multiple partners (IoU > 0).

    For each image, counts overlapping GT→pred (1:n) and pred→GT (n:1) links,
    then draws a split bar chart (GT matches on the right, prediction matches
    on the left).

    Args:
        overlap_df: Per-image overlap table as produced by ``get_overlap_df``;
            must contain columns ``iou_matrix``, ``n_gt``, and ``n_pred``.
        title: Plot title.
    """
    offset = 0.5

    # per-GT number of matched predictions (1:n) and per-prediction number of matched GTs (n:1)
    gt_match_counts: list[int] = []
    pred_match_counts: list[int] = []

    for row in overlap_df.itertuples(index=False):
        iou_matrix = row.iou_matrix
        n_gt = row.n_gt
        n_pred = row.n_pred

        if n_gt == 0:
            pred_match_counts.extend([0] * n_pred)
            continue

        if n_pred == 0:
            gt_match_counts.extend([0] * n_gt)
            continue

        hit_matrix = iou_matrix > 0

        gt_match_counts.extend(hit_matrix.sum(dim=1).tolist())
        pred_match_counts.extend(hit_matrix.sum(dim=0).tolist())

    gt_degree_counts = Counter(gt_match_counts)
    pred_degree_counts = Counter(pred_match_counts)

    gt_miss_count = gt_degree_counts.get(0, 0)
    gt_hit_count = sum(count for degree, count in gt_degree_counts.items() if degree > 0)
    pred_miss_count = pred_degree_counts.get(0, 0)
    pred_hit_count = sum(count for degree, count in pred_degree_counts.items() if degree > 0)

    color_gt_miss = "crimson"
    color_gt_hit = "forestgreen"
    color_pred_miss = "darkorange"
    color_pred_hit = "steelblue"

    gt_bars = {degree + offset: count for degree, count in gt_degree_counts.items()}
    pred_bars = {-(degree + offset): count for degree, count in pred_degree_counts.items()}

    gt_positions = sorted(gt_bars)
    pred_positions = sorted(pred_bars)
    bar_width = 0.8

    for pos in gt_positions:
        degree = int(round(pos - offset))
        plt.bar(
            pos,
            gt_bars[pos],
            width=bar_width,
            color=color_gt_miss if degree == 0 else color_gt_hit,
            edgecolor="black",
            alpha=0.7,
        )

    for pos in pred_positions:
        degree = int(round(-pos - offset))
        plt.bar(
            pos,
            pred_bars[pos],
            width=bar_width,
            color=color_pred_miss if degree == 0 else color_pred_hit,
            edgecolor="black",
            alpha=0.7,
        )

    plt.axvline(0, color="black", linewidth=0.8)
    plt.grid(axis="y", linestyle="--")
    plt.xlabel("Matching Case")
    plt.ylabel("Frequency")
    tick_positions = np.concatenate([gt_positions, pred_positions])
    tick_labels = [
        f"{abs(int(round(v + offset)))}:1" if v < 0 else f"1:{int(round(v - offset))}"
        for v in tick_positions
    ]
    plt.xticks(tick_positions, tick_labels, rotation=45, fontsize=9)
    plt.title(title)

    max_pos = max([abs(k) for k in gt_positions + pred_positions], default=offset)
    plt.xlim(-max_pos - bar_width, max_pos + bar_width)

    legend_handles = [
        Patch(facecolor=color_gt_hit, edgecolor="black", alpha=0.7, label=f"GT Hits (1:n, n>0): {gt_hit_count}"),
        Patch(facecolor=color_gt_miss, edgecolor="black", alpha=0.7, label=f"GT Misses (1:0): {gt_miss_count}"),
        Patch(facecolor=color_pred_miss, edgecolor="black", alpha=0.7, label=f"Prediction Misses (0:1): {pred_miss_count}"),
        Patch(facecolor=color_pred_hit, edgecolor="black", alpha=0.7, label=f"Prediction Hits (n:1, n>0): {pred_hit_count}"),
    ]
    plt.legend(handles=legend_handles, fontsize=9)


In [ ]:
def build_validation_gt_df():
    rows = []
    for record in val_records.itertuples(index=False):
        image_id = str(record.image_id)
        annotations = coco.loadAnns(coco.getAnnIds(imgIds=[record.image_id]))
        for index, annotation in enumerate(annotations, start=1):
            binary = coco.annToMask(annotation).astype(np.uint8)
            rle = mask_utils.encode(np.asfortranarray(binary))
            rows.append({'filament_id': f'{image_id}_{index}',
                         'segmentation_rle': rle['counts'].decode('utf-8')})
    return pd.DataFrame(rows, columns=['filament_id', 'segmentation_rle'])

def prediction_dataframe_for_config(cache_dir, split_records, config):
    rows = []
    for record in split_records:
        pred = read_prediction_cache(cache_dir, record['file_name'])
        instances = postprocess_predictions(pred, config)
        prefix = Path(record['file_name']).stem
        for index, instance in enumerate(instances, start=1):
            rows.append({'filament_id': f'{prefix}_{index}',
                         'segmentation_rle': encode_mask_rle(instance['mask'])})
        del pred, instances
    return pd.DataFrame(rows, columns=['filament_id', 'segmentation_rle'])

def _official_row_counts(row):
    matrix, n_gt, n_pred = row.iou_matrix, int(row.n_gt), int(row.n_pred)
    if n_gt == 0:
        return {'tp': 0, 'fp': n_pred, 'fn': 0, 'iou_sum': 0.0, 'gt_multi': 0, 'pred_multi': 0}
    if n_pred == 0:
        return {'tp': 0, 'fp': 0, 'fn': n_gt, 'iou_sum': 0.0, 'gt_multi': 0, 'pred_multi': 0}
    hits = matrix > OFFICIAL_IOU_THRESHOLD
    overlap = matrix > 0
    return {
        'tp': int(hits.sum().item()), 'fp': int(fp_count_hit(hits)), 'fn': int(fn_count_hit(hits)),
        'iou_sum': float(matrix[hits].sum().item()),
        'gt_multi': int((overlap.sum(dim=1) > 1).sum().item()),
        'pred_multi': int((overlap.sum(dim=0) > 1).sum().item()),
    }

def summarize_official_overlap(overlap_df):
    pq = float(get_pq_score(overlap_df))
    total = Counter()
    for row in overlap_df.itertuples(index=False):
        total.update(_official_row_counts(row))
    tp, fp, fn = total['tp'], total['fp'], total['fn']
    denominator = tp + 0.5 * fp + 0.5 * fn
    return {
        'PQ': pq,
        'SQ': float(total['iou_sum'] / tp) if tp else 0.0,
        'RQ': float(tp / denominator) if denominator else 0.0,
        'TP': tp, 'FP': fp, 'FN': fn,
        'matched_iou_sum': float(total['iou_sum']),
        'one_to_many_iou_gt0': int(total['gt_multi']),
        'many_to_one_iou_gt0': int(total['pred_multi']),
    }

def official_per_image_summary(overlap_df):
    if overlap_df.empty:
        return pd.DataFrame()
    work = overlap_df.copy()
    work['physical_image_id'] = work['annotator_image'].map(lambda value: value.split('-', maxsplit=1)[1])
    output = []
    for image_id, group in work.groupby('physical_image_id', sort=False):
        metrics = summarize_official_overlap(group)
        output.append({
            'physical_image_id': image_id, 'PQ': metrics['PQ'], 'SQ': metrics['SQ'], 'RQ': metrics['RQ'],
            'TP': metrics['TP'], 'FP': metrics['FP'], 'FN': metrics['FN'],
            'predictions_across_annotators': int(group['n_pred'].sum()),
            'gt_across_annotators': int(group['n_gt'].sum()),
        })
    return pd.DataFrame(output)

gt_df = build_validation_gt_df()
if gt_df.empty:
    raise ValueError('No validation GT rows were generated.')
LOGGER.info('Validation GT RLE rows: %s',len(gt_df))
print(gt_df.head())

## 11. Experiment runner
Each experiment rereads raw prediction cache entries, post-processes them, encodes instances in the official RLE format, and evaluates with the official functions. No inference occurs in this loop.

In [ ]:
def evaluate_experiment(experiment):
    start = time.perf_counter()
    pred_df = prediction_dataframe_for_config(VAL_CACHE_DIR, val_cache_records, experiment['config'])
    overlap_df = get_overlap_df(gt_df, pred_df)
    metrics = summarize_official_overlap(overlap_df)
    per_image = official_per_image_summary(overlap_df)
    row = {
        'experiment': experiment['name'],
        'confidence': float(experiment['config'].get('confidence', 0)),
        'min_area': int(experiment['config'].get('min_instance_area', 0)),
        'component_min_area': int(experiment['config'].get('component_min_area', COMPONENT_MIN_AREA)),
        'merge_distance': experiment['config'].get('merge', {}).get('max_endpoint_distance', np.nan),
        'direction_threshold': experiment['config'].get('merge', {}).get('min_direction_cosine', np.nan),
        'max_width_ratio': experiment['config'].get('merge', {}).get('max_width_ratio', np.nan),
        'gap_bridge_distance': experiment['config'].get('gap_bridge', {}).get('max_endpoint_distance', np.nan),
        'morphology': json.dumps(experiment['config'].get('morphology', {}), sort_keys=True),
        'PQ': metrics['PQ'], 'SQ': metrics['SQ'], 'RQ': metrics['RQ'],
        'TP': metrics['TP'], 'FP': metrics['FP'], 'FN': metrics['FN'],
        'one_to_many_iou_gt0': metrics['one_to_many_iou_gt0'],
        'many_to_one_iou_gt0': metrics['many_to_one_iou_gt0'],
        'runtime_seconds': time.perf_counter() - start,
        'config_json': json.dumps(experiment['config'], sort_keys=True),
    }
    del pred_df
    return row, overlap_df, per_image

experiment_rows = []
experiment_per_image = {}
best_overlap_df, best_pq_seen = None, -1.0
for experiment in tqdm(EXPERIMENTS, desc='Post-processing experiments', disable=not DEBUG):
    LOGGER.debug('Evaluating experiment %s with config %s',experiment['name'],json.dumps(experiment['config'],sort_keys=True))
    row, overlap, per_image = evaluate_experiment(experiment)
    experiment_rows.append(row)
    LOGGER.info('Experiment %s complete | PQ=%.5f | %.1f sec',row['experiment'],row['PQ'],row['runtime_seconds'])
    experiment_per_image[experiment['name']] = per_image
    if row['PQ'] > best_pq_seen:
        best_pq_seen = row['PQ']
        best_overlap_df = overlap
    pd.DataFrame(experiment_rows).sort_values('PQ', ascending=False).to_csv(RESULTS_PATH, index=False)
    del overlap
    gc.collect()

results_df = pd.DataFrame(experiment_rows).sort_values('PQ', ascending=False).reset_index(drop=True)
results_df.to_csv(RESULTS_PATH, index=False)
display(results_df.drop(columns=['config_json']).round(4))
LOGGER.info('Experiment results saved: %s',RESULTS_PATH)

## 12. Results and official overlap diagnostics
Select the highest global official PQ and save the config explicitly. Official IoU/Dice distributions and relation plots are retained for the best experiment.

In [ ]:
best_row = results_df.iloc[0]
best_record = next(x for x in EXPERIMENTS if x['name'] == best_row['experiment'])
BEST_CONFIG = {'name': best_record['name'], 'config': deepcopy(best_record['config'])}
BEST_PER_IMAGE = experiment_per_image[BEST_CONFIG['name']].copy()
BEST_PER_IMAGE.to_csv(BEST_PER_IMAGE_PATH, index=False)
with open(BEST_CONFIG_PATH, 'w', encoding='utf-8') as f:
    json.dump(BEST_CONFIG, f, indent=2, sort_keys=True)

print('BEST EXPERIMENT')
print('----------------')
print(f"Name: {BEST_CONFIG['name']}")
print(f"PQ: {best_row['PQ']:.6f} | SQ: {best_row['SQ']:.6f} | RQ: {best_row['RQ']:.6f}")
print(f"TP: {int(best_row['TP'])} | FP: {int(best_row['FP'])} | FN: {int(best_row['FN'])}")
print(f'Config: {BEST_CONFIG_PATH}')

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
plt.sca(axes[0]); plot_distribution(best_overlap_df['iou_matrix'], 'Official IoU distribution', 'IoU')
plt.sca(axes[1]); plot_distribution(best_overlap_df['dice_matrix'], 'Official Dice distribution', 'Dice')
plt.tight_layout(); plt.show()
plt.figure(figsize=(8, 4.5))
plot_m2n_counts(best_overlap_df, 'Official many-to-one / one-to-many overlap relations')
plt.tight_layout(); plt.show()

## 13. Optional baseline-checkpoint comparison

If a baseline checkpoint is supplied, run it through the same CPU tile inference, cache, stitching configuration, validation images and official evaluator. This is optional; no baseline weights are bundled with the workspace.

In [ ]:
if BASELINE_COMPARISON_PATH is None:
    BASELINE_COMPARISON_PATH = KAGGLE_WORKING / 'baseline_comparison.csv'

BASELINE_COMPARISON = None
if BASELINE_CHECKPOINT_PATH is None:
    print('Baseline comparison skipped: set BASELINE_CHECKPOINT_PATH to enable it.')
else:
    baseline_path = Path(BASELINE_CHECKPOINT_PATH).expanduser().resolve()
    if not baseline_path.is_file():
        raise FileNotFoundError(f'Configured baseline checkpoint missing: {baseline_path}')
    baseline_stat = baseline_path.stat()
    baseline_identity = {'path': str(baseline_path), 'size_bytes': int(baseline_stat.st_size),
                         'mtime_ns': int(baseline_stat.st_mtime_ns)}
    cache_tag = hashlib.sha1(json.dumps(baseline_identity, sort_keys=True).encode()).hexdigest()[:12]
    old_state = (TRAINED_CHECKPOINT, CHECKPOINT_IDENTITY, model, CACHE_WRITE_ROOT, CACHE_READ_ROOT)
    try:
        TRAINED_CHECKPOINT = baseline_path
        CHECKPOINT_IDENTITY = baseline_identity
        model = None
        CACHE_WRITE_ROOT = KAGGLE_WORKING / f'baseline_prediction_cache_{cache_tag}'
        CACHE_READ_ROOT = None
        baseline_inference_start = time.perf_counter()
        baseline_cache_dir = ensure_prediction_cache('val', val_cache_records)
        baseline_cache_seconds = time.perf_counter() - baseline_inference_start
        baseline_eval_start = time.perf_counter()
        baseline_pred_df = prediction_dataframe_for_config(baseline_cache_dir, val_cache_records, BASELINE_CONFIG)
        baseline_overlap = get_overlap_df(gt_df, baseline_pred_df)
        baseline_metrics = summarize_official_overlap(baseline_overlap)
        baseline_eval_seconds = time.perf_counter() - baseline_eval_start
        crop_row = results_df.iloc[0]
        BASELINE_COMPARISON = pd.DataFrame([
            {'model':'YOLO26s crop-trained','checkpoint':str(old_state[0]),
             'training_strategy':'annotation-specific 640x640 crops',
             'validation_split_id':split_identifier,'postprocessing':str(crop_row['experiment']),
             'PQ':float(crop_row['PQ']),'SQ':float(crop_row['SQ']),'RQ':float(crop_row['RQ']),
             'TP':int(crop_row['TP']),'FP':int(crop_row['FP']),'FN':int(crop_row['FN']),
             'inference_cache_seconds':float(elapsed_time),
             'postprocess_stitch_eval_seconds':float(crop_row['runtime_seconds'])},
            {'model':'Configured baseline checkpoint','checkpoint':str(baseline_path),
             'training_strategy':'external baseline checkpoint','validation_split_id':split_identifier,
             'postprocessing':'baseline configuration','PQ':baseline_metrics['PQ'],
             'SQ':baseline_metrics['SQ'],'RQ':baseline_metrics['RQ'],'TP':baseline_metrics['TP'],
             'FP':baseline_metrics['FP'],'FN':baseline_metrics['FN'],
             'inference_cache_seconds':float(baseline_cache_seconds),
             'postprocess_stitch_eval_seconds':float(baseline_eval_seconds)},
        ])
        BASELINE_COMPARISON.to_csv(BASELINE_COMPARISON_PATH,index=False)
        display(BASELINE_COMPARISON.round(4))
        print('Comparison saved:',BASELINE_COMPARISON_PATH)
    finally:
        TRAINED_CHECKPOINT, CHECKPOINT_IDENTITY, model, CACHE_WRITE_ROOT, CACHE_READ_ROOT = old_state

## 14. Validation visualizations and diagnostic examples
Comparison panels show original image, source-compatible baseline, selected strategy, and the union of GT masks for display. Evaluation still scores each annotator separately; the GT union is only for visual context.

The scan searches for a merge config that raises per-image official PQ while reducing instance count, one that lowers PQ while reducing count, a low-confidence component removed at 0.60 with no official-threshold GT hit, and a removed component with a GT hit. If the split has no example in a category, the notebook reports that.

In [ ]:
def get_gt_masks_for_file(file_name):
    masks = []
    for image_id in val_records.loc[val_records.file_name == file_name, 'image_id'].tolist():
        for ann_id in coco.getAnnIds(imgIds=[image_id]):
            ann = coco.loadAnns([ann_id])[0]
            masks.append(coco.annToMask(ann).astype(bool))
    return masks

def _overlay(image, instances, title):
    rgb = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB).astype(np.float32)
    palette = plt.get_cmap('tab20')
    for i, item in enumerate(instances):
        mask = np.asarray(item['mask'], dtype=bool)
        color = (np.asarray(palette(i % 20)[:3]) * 255).astype(np.float32)
        rgb[mask] = 0.45 * rgb[mask] + 0.55 * color
        contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if contours:
            cv2.drawContours(rgb, contours, -1, color.tolist(), 1)
    plt.imshow(np.clip(rgb, 0, 255).astype(np.uint8))
    plt.title(title); plt.axis('off')

def render_validation_comparison(file_name, config=None, caption='Validation comparison'):
    image = cv2.imread(str(TRAIN_IMAGE_DIR / file_name), cv2.IMREAD_GRAYSCALE)
    if image is None:
        raise OSError(f'Could not read validation image: {file_name}')
    raw = read_prediction_cache(VAL_CACHE_DIR, file_name)
    baseline = postprocess_predictions(raw, BASELINE_CONFIG)
    cfg = config or BEST_CONFIG['config']
    if isinstance(cfg, dict) and 'config' in cfg:
        cfg = cfg['config']
    processed = postprocess_predictions(raw, cfg)
    gt_masks = get_gt_masks_for_file(file_name)
    fig, axes = plt.subplots(1, 4, figsize=(18, 5))
    plt.sca(axes[0]); plt.imshow(image, cmap='gray'); plt.title('Original'); plt.axis('off')
    plt.sca(axes[1]); _overlay(image, baseline, f'Baseline ({len(baseline)})')
    plt.sca(axes[2]); _overlay(image, processed, f'Processed ({len(processed)})')
    plt.sca(axes[3]); _overlay(image, [_instance(m, 1, 0) for m in gt_masks if m.any()], 'GT union for display')
    fig.suptitle(f'{caption}: {file_name}', y=1.02)
    plt.tight_layout(); plt.show()
    del image, raw, baseline, processed, gt_masks
    gc.collect()

stem_to_validation_file = {Path(r['file_name']).stem: r['file_name'] for r in val_cache_records}

def select_merge_examples():
    baseline = experiment_per_image['baseline'].set_index('physical_image_id')
    improved, harmed = None, None
    for exp in EXPERIMENTS:
        if not exp['config'].get('merge', {}).get('enabled', False):
            continue
        current = experiment_per_image[exp['name']].set_index('physical_image_id')
        pair = baseline[['PQ', 'predictions_across_annotators']].join(
            current[['PQ', 'predictions_across_annotators']], how='inner',
            lsuffix='_base', rsuffix='_merged')
        pair['delta'] = pair['PQ_merged'] - pair['PQ_base']
        pair = pair[pair['predictions_across_annotators_merged'] < pair['predictions_across_annotators_base']]
        if pair.empty:
            continue
        positives, negatives = pair[pair.delta > 0], pair[pair.delta < 0]
        if not positives.empty:
            row = positives.sort_values('delta', ascending=False).iloc[0]
            candidate = {'image_id': row.name, 'experiment': exp['name'], 'delta': float(row.delta)}
            if improved is None or candidate['delta'] > improved['delta']:
                improved = candidate
        if not negatives.empty:
            row = negatives.sort_values('delta').iloc[0]
            candidate = {'image_id': row.name, 'experiment': exp['name'], 'delta': float(row.delta)}
            if harmed is None or candidate['delta'] < harmed['delta']:
                harmed = candidate
    return improved, harmed

def _max_iou_to_any_gt(mask, targets):
    area = int(mask.sum())
    best = 0.0
    for target in targets:
        gt_area = int(target.sum())
        if area and gt_area:
            intersection = int(np.logical_and(mask, target).sum())
            best = max(best, intersection / (area + gt_area - intersection))
    return best

def find_confidence_examples(low=0.05, high=0.60):
    fp_example, useful_example = None, None
    low_cfg, high_cfg = make_config(confidence=low), make_config(confidence=high)
    for record in tqdm(val_cache_records, desc='Finding confidence examples'):
        name = record['file_name']
        raw = read_prediction_cache(VAL_CACHE_DIR, name)
        low_instances = postprocess_predictions(raw, low_cfg)
        _ = postprocess_predictions(raw, high_cfg)
        targets = get_gt_masks_for_file(name)
        for item in low_instances:
            if item['confidence'] >= high:
                continue
            score = _max_iou_to_any_gt(item['mask'], targets)
            candidate = {'file_name': name, 'max_gt_iou': float(score)}
            if score <= OFFICIAL_IOU_THRESHOLD and fp_example is None:
                fp_example = candidate
            if score > OFFICIAL_IOU_THRESHOLD and useful_example is None:
                useful_example = candidate
        del raw, low_instances, targets
        if fp_example and useful_example:
            break
    return fp_example, useful_example

merge_improved, merge_harmed = select_merge_examples()
if merge_improved:
    print('Merge improved example:', merge_improved)
    exp = next(x for x in EXPERIMENTS if x['name'] == merge_improved['experiment'])
    render_validation_comparison(stem_to_validation_file[merge_improved['image_id']], exp['config'],
                                 f'Merge raised per-image official PQ by {merge_improved["delta"]:.4f}')
else:
    print('No merge-improved example with a reduced instance count was found.')
if merge_harmed:
    print('Potential wrong-merge example:', merge_harmed)
    exp = next(x for x in EXPERIMENTS if x['name'] == merge_harmed['experiment'])
    render_validation_comparison(stem_to_validation_file[merge_harmed['image_id']], exp['config'],
                                 f'Merge lowered per-image official PQ by {abs(merge_harmed["delta"]):.4f}')
else:
    print('No merge-harmed example with a reduced instance count was found.')

confidence_fp, confidence_useful = find_confidence_examples()
if confidence_fp:
    print('Confidence removed an official-threshold FP candidate:', confidence_fp)
    render_validation_comparison(confidence_fp['file_name'], make_config(confidence=0.60), 'Confidence filtering at 0.60')
else:
    print('No removed low-confidence FP candidate was found.')
if confidence_useful:
    print('Confidence also removed a GT-hit filament:', confidence_useful)
    render_validation_comparison(confidence_useful['file_name'], make_config(confidence=0.60), 'Useful filament removed at 0.60')
else:
    print('No removed GT-hit example was found.')

if val_cache_records:
    render_validation_comparison(val_cache_records[0]['file_name'], BEST_CONFIG, 'Best validation configuration')

def render_tile_stitching_diagnostic(file_name):
    image=cv2.imread(str(TRAIN_IMAGE_DIR/file_name),cv2.IMREAD_GRAYSCALE)
    raw=read_prediction_cache(VAL_CACHE_DIR,file_name)
    origins=sorted({tuple(item['tile_origin']) for item in raw['instances']},key=lambda xy:(xy[1],xy[0]))
    if origins:
        counts=Counter(tuple(item['tile_origin']) for item in raw['instances'])
        chosen=sorted(origins,key=lambda xy:(-counts[xy],xy[1],xy[0]))[0]
    else:
        chosen=(tile_starts(image.shape[1])[0],tile_starts(image.shape[0])[0])
    tile=extract_tile(image,*chosen)
    local=[item for item in raw['instances'] if tuple(item['tile_origin'])==chosen]
    stitched=stitch_tile_predictions(raw['instances'],raw['image_shape'],BASELINE_CONFIG['stitch'])
    final=postprocess_predictions(raw,BASELINE_CONFIG)
    fig,axes=plt.subplots(1,4,figsize=(20,5))
    axes[0].imshow(image,cmap='gray');axes[0].set_title('Original with inference tiles')
    for x in tile_starts(image.shape[1]):axes[0].axvline(x,color='yellow',linewidth=.5);axes[0].axvline(x+TILE_SIZE,color='yellow',linewidth=.5)
    for y in tile_starts(image.shape[0]):axes[0].axhline(y,color='yellow',linewidth=.5);axes[0].axhline(y+TILE_SIZE,color='yellow',linewidth=.5)
    axes[0].axis('off')
    axes[1].imshow(tile,cmap='gray');axes[1].set_title(f'Raw predictions in tile {chosen}')
    for item in local:
        if item['mask'].any():axes[1].contour(item['mask'].astype(np.uint8),levels=[.5],linewidths=.7)
    axes[1].axis('off')
    plt.sca(axes[2]);_overlay(image,stitched,f'Stitched tile instances ({len(stitched)})')
    plt.sca(axes[3]);_overlay(image,final,f'Final baseline ({len(final)})')
    fig.suptitle(file_name);plt.tight_layout();plt.show()
    del image,raw,stitched,final;gc.collect()

if val_cache_records:
    render_tile_stitching_diagnostic(val_cache_records[0]['file_name'])


## 15. Freeze the selected validation configuration
The best config is persisted as JSON. The test path below reads this saved value and does not tune on test data.

In [ ]:
with open(BEST_CONFIG_PATH, 'r', encoding='utf-8') as f:
    BEST_CONFIG = json.load(f)
LOGGER.debug('Selected post-processing config: %s',json.dumps(BEST_CONFIG,indent=2,sort_keys=True))
LOGGER.info('Selected validation PQ: %.6f',float(results_df.iloc[0]['PQ']))

## 16. Test inference
Run only after selecting a configuration on validation. A complete matching test cache is reused; otherwise each test image is inferred once using the same fixed YOLO settings. No test ground truth is loaded.

In [ ]:
test_cache_start=time.perf_counter()
TEST_CACHE_DIR = ensure_prediction_cache('test', test_cache_records)
test_cache_seconds=time.perf_counter()-test_cache_start
LOGGER.info('Test cache ready: %s | elapsed: %.1f min (includes cache reuse if present)',TEST_CACHE_DIR,test_cache_seconds/60)

## 17. Apply post-processing and create submission
Post-process cached test predictions with the frozen config. The required CSV columns and image-stem instance IDs follow the existing polygon inference notebook.

In [ ]:
def create_submission_from_cache(output_path=SUBMISSION_PATH):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    seen, row_count = set(), 0
    with output_path.open('w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['filament_id', 'segmentation_rle'])
        for record in tqdm(test_cache_records, desc='Post-process test cache', disable=not DEBUG):
            LOGGER.debug('Post-processing cached test image %s',record['file_name'])
            pred = read_prediction_cache(TEST_CACHE_DIR, record['file_name'])
            instances = postprocess_predictions(pred, BEST_CONFIG['config'])
            prefix = Path(record['file_name']).stem
            for number, instance in enumerate(instances, start=1):
                filament_id = f'{prefix}_{number}'
                if filament_id in seen:
                    raise ValueError(f'Duplicate filament ID: {filament_id}')
                seen.add(filament_id)
                writer.writerow([filament_id, encode_mask_rle(instance['mask'])])
                row_count += 1
            del pred, instances
    return {'path': output_path, 'rows_written': row_count}

submission_info = create_submission_from_cache()
LOGGER.info('Wrote %s with %s instance rows',submission_info['path'],submission_info['rows_written'])

## 18. Final submission validation
The audit checks exact column order, required IDs, duplicates, test-image membership, exclusion of train/validation IDs, and decodes every RLE to confirm its original dimensions and nonempty mask. Per-test-image instance counts are saved to CSV.

In [ ]:
def audit_submission(path, expected_records):
    expected_columns = ['filament_id', 'segmentation_rle']
    expected_stems = {Path(r['file_name']).stem for r in expected_records}
    train_stems = {Path(name).stem for name in train_files}
    val_stems = {Path(name).stem for name in val_files}
    seen, per_image = set(), Counter({stem: 0 for stem in expected_stems})
    row_count = null_count = empty_count = duplicate_count = 0
    decode_errors = wrong_shape = empty_masks = malformed_ids = 0
    with Path(path).open('r', newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        columns = reader.fieldnames
        for row in reader:
            row_count += 1
            if any(value is None for value in row.values()):
                null_count += 1
            fid, rle = row.get('filament_id'), row.get('segmentation_rle')
            if not fid or not rle:
                empty_count += 1
                continue
            if fid in seen:
                duplicate_count += 1
            seen.add(fid)
            if '_' not in fid:
                malformed_ids += 1
                continue
            stem = fid.rsplit('_', 1)[0]
            if stem not in expected_stems:
                continue
            per_image[stem] += 1
            try:
                decoded = mask_utils.decode({'size': [EVALUATION_HEIGHT, EVALUATION_WIDTH], 'counts': rle})
                if decoded.shape != (EVALUATION_HEIGHT, EVALUATION_WIDTH):
                    wrong_shape += 1
                elif not decoded.any():
                    empty_masks += 1
            except Exception:
                decode_errors += 1
    actual_stems = {fid.rsplit('_', 1)[0] for fid in seen if '_' in fid}
    stems_ok = actual_stems <= expected_stems
    train_val_excluded = not (actual_stems & (train_stems | val_stems))
    valid = (columns == expected_columns and null_count == 0 and empty_count == 0 and
             duplicate_count == 0 and decode_errors == 0 and wrong_shape == 0 and
             empty_masks == 0 and malformed_ids == 0 and stems_ok and train_val_excluded)
    counts_path = KAGGLE_WORKING / 'submission_instance_counts.csv'
    pd.DataFrame([{'file_stem': stem, 'instance_count': per_image[stem]}
                  for stem in sorted(expected_stems)]).to_csv(counts_path, index=False)
    return {'columns': columns, 'rows': row_count, 'null_rows': null_count,
            'empty_id_or_rle_rows': empty_count, 'duplicate_ids': duplicate_count,
            'rle_decode_errors': decode_errors, 'wrong_mask_shapes': wrong_shape,
            'empty_masks': empty_masks, 'malformed_ids': malformed_ids,
            'ids_reference_test_images': stems_ok, 'train_validation_ids_excluded': train_val_excluded,
            'instances_per_test_image_csv': str(counts_path), 'valid': valid}

submission_audit = audit_submission(SUBMISSION_PATH, test_cache_records)
LOGGER.info('Submission audit: %s',json.dumps(submission_audit,sort_keys=True))
if not submission_audit['valid']:
    raise ValueError('Submission audit failed.')
LOGGER.info('Validated all submission rows: %s',SUBMISSION_PATH)

## 19. Reproducibility notes
- Validation parameter selection uses only the filename-grouped validation split and official evaluator.
- Validation and test caches are separate, record checkpoint identity, and store tile-local masks plus tile coordinates for restartable 640-pixel inference.
- Post-processing experiments do not load YOLO or reread images; they read cached masks and metadata.
- To reuse cache between Kaggle sessions, attach the cache output as a dataset and set CACHE_READ_ROOT. Keep CACHE_WRITE_ROOT writable for missing entries.
- Outputs: postprocessing_results.csv, postprocessing_best_config.json, postprocessing_best_per_image.csv, submission_instance_counts.csv, and submission.csv.
- The official ID parser splits at the first underscore; the supplied competition image stems follow the no-underscore convention.